# Regrid reanalyses to 1-degree grid

In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import xagg as xa
import xesmf as xe
import os
import glob
import re
import shutil

from tqdm.notebook import tqdm

from funcs_support import get_filepaths,get_params,utility_save
from funcs_aux import _find_main_variable

dir_list = get_params()

In [3]:
from distributed import Client
client = Client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /node/crhtc78.hpc.ucar.edu/14966/proxy/8787/status,
Dashboard: /node/crhtc78.hpc.ucar.edu/14966/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 144.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33433,Workers: 0
Dashboard: /node/crhtc78.hpc.ucar.edu/14966/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45575,Total threads: 3
Dashboard: /node/crhtc78.hpc.ucar.edu/14966/proxy/33133/status,Memory: 24.00 GiB
Nanny: tcp://127.0.0.1:41633,


2026-10-09 13:43:30,868 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 1f6baed4f0d090b028bf83c94d776a51 initialized by task ('rechunk-merge-rechunk-transfer-53a9a587945da998501880271683629f', 25, 0, 0, 37, 7, 0) executed on worker tcp://127.0.0.1:42781
2026-10-09 13:43:31,356 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 9fb080dccc89ca671549d27f7db11d3f initialized by task ('rechunk-merge-rechunk-transfer-53a9a587945da998501880271683629f', 23, 0, 0, 34, 2, 0) executed on worker tcp://127.0.0.1:45543
2026-10-09 13:43:35,295 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 62beb0ae87e0320b48c89f3672242aab initialized by task ('rechunk-merge-rechunk-transfer-53a9a587945da998501880271683629f', 21, 0, 0, 31, 7, 0) executed on worker tcp://127.0.0.1:46659
2026-10-09 13:43:35,509 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 9fb080dccc89ca671549d27f7db11d3f deactivated due to stimulus 'task-finished-1791575015.5089397'
2026-10-09 13:43:35,88

In [4]:
df = get_filepaths()

In [5]:
params_subset = {'lat':slice(-56,86),'lon':slice(-180,180)}
# Get reference 1x1 grid
ref_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.5,
														 params_subset['lat'].stop-0.4)),
								'lon':(['lon'],np.arange(params_subset['lon'].start+0.5,
														 params_subset['lon'].stop-0.4))})


In [6]:
modlist = ['UDel','ERA5-025','MERRA2','JRA-3Q','GMFD','PRISM']
varlist = ['tas','tasmax','pr','prcorr']

In [7]:
df = df.query('run == "reanalysis" or run == "obs"')
df = df.iloc[[mod in modlist for mod in df.model.values],:]
df = df.iloc[[var in varlist for var in df.varname.values],:]

In [11]:
'1deg/' in output_fn

True

In [13]:
overwrite = True

for row in tqdm(df.iterrows(),total=len(df)):
    output_fn = re.sub(row[1]['model'],row[1]['model']+'-1deg',row[1]['path'])
    if '1deg/' not in output_fn:
        print(f'Issue renaming {output_fn} to reflect 1-deg model rename, skipping...')
        continue
    print(f'Processing {output_fn}...')
    if overwrite or (not os.path.exists(output_fn)):
        if row[1]['filetype'] == 'zarr':
            ds = xr.open_zarr(row[1]['path'])

            # Rechunk into temporary store
            print(f'Rechunking...')
            chunk_sizes = ds[_find_main_variable(ds)].chunksizes
            ngridcells = ds.sizes['lat']*ds.sizes['lon']
            if ngridcells > 100000:
                time_chunksize = 50
            elif ngridcells > 10000:
                time_chunksize = 500
            else:
                time_chunksizes = 5000
            ds = ds.chunk({'lat':-1,'lon':-1,'time':time_chunksize})

            # Save temporary rechunked file
            fn_rechk = re.sub(r'\.zarr','_RECHUNK-TMP.zarr',row[1]['path'])
            utility_save(ds.drop_encoding(),fn_rechk,zarr_mode='w',raw_overwrite_flag=True)

            # Now, load the temporary rechunked file
            ds = xr.open_zarr(fn_rechk)
        else:
            ds = xr.open_dataset(row[1]['path'])
        if 'pr' in row[1]['varname']:
            rgrd_typ = 'conservative'
        else:
            rgrd_typ = 'bilinear'

        if (((ds.lon.max() > 175) and (ds.lon.min() < -175)) or 
            ((ds.lon.max() > 355) and (ds.lon.min() < 5))):
            periodic = True
        else:
            periodic = False
        print(f'Regridding...')
        rgrd = xe.Regridder(ds,ref_grid,rgrd_typ,ignore_degenerate=True,periodic=periodic)
        ds = rgrd(ds)
        
        ds.attrs['SOURCE'] = '0a_create_1deg_reanalyses.ipynb'
        ds.attrs['DESCRIPTION'] = f'Data, regridded using {rgrd_typ} algorithm to 1-degree BCD-ME grid from {re.split(r'\/',row[1]['path'])[-1]}'
        if row[1]['filetype'] == 'zarr':
            ds = ds.drop_encoding()
            utility_save(ds,output_fn,zarr_mode='w',raw_overwrite_flag=True)
        else:
            utility_save(ds,output_fn,raw_overwrite_flag=True)

        # Remove temporary rechunked file
        if row[1]['filetype'] == 'zarr':
            print(f'Removing temporary file...')
            shutil.rmtree(fn_rechk)
    else:
        print(output_fn+' already processed, skipped.')
# ~ 30 minutes for all but PRISM on 18 cores, about 1.5 minutes extra for PRISM

  0%|          | 0/21 [00:00<?, ?it/s]

Processing /glade/derecho/scratch/schwarzwald/climate_raw_derecho/PRISM-1deg/pr_day_PRISM-1deg_historical_obs_1981010101-20260630_CONUS.zarr...
Rechunking...
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/PRISM/pr_day_PRISM_historical_obs_1981010101-20260630_CONUS_RECHUNK-TMP.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/climate_raw_derecho/PRISM/pr_day_PRISM_historical_obs_1981010101-20260630_CONUS_RECHUNK-TMP.zarr saved!
Regridding...
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/PRISM-1deg/pr_day_PRISM-1deg_historical_obs_1981010101-20260630_CONUS.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 21.67 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


...

/glade/derecho/scratch/schwarzwald/climate_raw_derecho/GMFD-1deg/pr_day_GMFD-1deg_historical_reanalysis_19570101-20101231.zarr saved!
Removing temporary file...
Processing /glade/derecho/scratch/schwarzwald/climate_raw_derecho/GMFD-1deg/pr_day_GMFD-1deg_historical_reanalysis_19570101-20101231_HoAfrica.zarr...
Rechunking...


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/climate_raw_derecho/GMFD/pr_day_GMFD_historical_reanalysis_19570101-20101231_HoAfrica_RECHUNK-TMP.zarr saved!
Regridding...


/glade/u/home/schwarzwald/.conda/envs/impact_uncert/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/climate_raw_derecho/GMFD-1deg/pr_day_GMFD-1deg_historical_reanalysis_19570101-20101231_HoAfrica.zarr saved!
Removing temporary file...
